# ⚡ Snippy: Gemma 3 270M Fine-Tuning & LiteRT Export

This notebook is specifically tailored for **Google Colab (T4 GPU / Python 3.12)**.
It fine-tunes **Gemma 3 270M** (`unsloth/gemma-3-270m-it`) as **Snippy** — an in-browser JavaScript Code Snippet & Tool Calling Agent — and exports the model to Google LiteRT format for browser WebGPU deployment.

## Step 1: Environment Setup
We upgrade `protobuf>=6.31.1` required by `litert_lm_builder` and set `USE_TORCHVISION=0`.

In [ ]:
import os
os.environ["USE_TORCHVISION"] = "0"

# Upgrade protobuf to fix litert_lm_builder VersionError
!pip install -q -U "protobuf>=6.31.1"
!pip install -q -U torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers peft datasets litert-torch litert-lm

## Step 2: Fetch Snippy Training Dataset
Fetch `snippy_dataset.json` containing 50+ generic tool calling instruction examples.

In [ ]:
import json
import os
from datasets import Dataset

DATASET_PATH = "snippy_dataset.json"
if not os.path.exists(DATASET_PATH):
    !wget -q https://raw.githubusercontent.com/xprilion/gemma3-270m-finetune-litert/main/snippy_dataset.json

with open(DATASET_PATH, "r") as f:
    sample_data = json.load(f)

dataset = Dataset.from_list(sample_data)
print(f"✅ Loaded {len(dataset)} training examples for Snippy in Colab!")

## Step 3: Fine-Tune Gemma 3 270M with PEFT / LoRA
Train Gemma 3 270M on Colab T4 GPU in seconds using PyTorch + HuggingFace Trainer.

In [ ]:
import os
os.environ["USE_TORCHVISION"] = "0"

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

MODEL_ID = "unsloth/gemma-3-270m-it"
OUTPUT_LORA_DIR = "./lora_adapter"
OUTPUT_MERGED_DIR = "./fine_tuned_gemma_merged"

# 1. Load Tokenizer & Base Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# 2. Configure LoRA for Gemma 3 270M
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
peft_model = get_peft_model(model, peft_config)

# 3. Tokenize Dataset
def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, max_length=256)

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 4. Fast, Stable PyTorch GPU Training
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    logging_steps=1,
    fp16=torch.cuda.is_available(),
    optim="adamw_torch",
    report_to="none"
)

trainer = Trainer(
    model=peft_model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator
)
trainer.train()

# Save LoRA Adapter
peft_model.save_pretrained(OUTPUT_LORA_DIR)
tokenizer.save_pretrained(OUTPUT_LORA_DIR)
print("✅ Fine-Tuning Complete! Gemma 3 270M LoRA Adapter Saved.")

## Step 4: Merge Adapter Weights into Base Model (GPU Accelerated)
Merge LoRA adapter weights using GPU (`device_map='auto'`).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# Load base model and adapter on GPU for fast merging
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
peft_model = PeftModel.from_pretrained(base_model, OUTPUT_LORA_DIR)
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained(OUTPUT_MERGED_DIR)
tokenizer.save_pretrained(OUTPUT_MERGED_DIR)
print("✅ Merged Model Saved on GPU!")

## Step 5: Convert Model to LiteRT Format (`litert-torch export_hf`)
Export model to `.litertlm` container with INT8 dynamic quantization.

In [ ]:
!litert-torch export_hf \
  ./fine_tuned_gemma_merged \
  ./litert_output \
  -b True \
  -q dynamic_int8

## Step 6: Extract WebGPU FlatBuffer & Download Converted Models
Extract the `TFL3` FlatBuffer (`model.tflite`) for WebGPU browser runtime and trigger direct file downloads in Colab.

In [ ]:
import os
from google.colab import files

SOURCE_MODEL = "./litert_output/model.litertlm"

if os.path.exists(SOURCE_MODEL):
    # Extract TFLite FlatBuffer for WebGPU browser engine
    with open(SOURCE_MODEL, "rb") as f:
        data = f.read()
    pos = data.find(b"TFL3")
    if pos != -1:
        tflite_data = data[pos - 4 :]
        with open("model.tflite", "wb") as f:
            f.write(tflite_data)
        print("✅ Extracted model.tflite (WebGPU FlatBuffer)!")
        files.download("model.tflite")
    
    print("✅ Downloading model.litertlm...")
    files.download(SOURCE_MODEL)
else:
    print(f"⚠️ Source model not found at {SOURCE_MODEL}")